# Field Development Digital Twin and Lifecycle

This notebook links the field-development lifecycle to the computational objects used in NeqSim. It generates reusable figures for Chapter 11.


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch11_field_development_building_blocks\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch11_field_development_building_blocks\figures
NeqSim Java bridge available: True


## Lifecycle Phases


In [2]:
phases = [
    ("Discovery", 1, "resource signal"),
    ("Appraisal", 2, "resource range"),
    ("Feasibility", 1, "screen concepts"),
    ("Concept select", 1, "rank alternatives"),
    ("FEED", 2, "define project"),
    ("Detailed design", 2, "freeze scope"),
    ("Construction", 3, "build and install"),
    ("Operations", 20, "optimize value"),
]
start = np.cumsum([0] + [p[1] for p in phases[:-1]])
durations = [p[1] for p in phases]
colors = plt.cm.Set3(np.linspace(0, 1, len(phases)))
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.barh([0] * len(phases), durations, left=start, color=colors, edgecolor="black")
for left, dur, (label, _, focus) in zip(start, durations, phases):
    ax.text(left + dur / 2, 0, label, ha="center", va="center", fontsize=9, fontweight="bold")
    ax.text(left + dur / 2, -0.22, focus, ha="center", va="center", fontsize=8)
ax.set_xlim(0, sum(durations))
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel("Approximate elapsed time (years)")
ax.set_title("Field Development Lifecycle and Main Decision Focus")
ax.grid(axis="x", alpha=0.25)
fig.savefig(FIGURES_DIR / "ch11_digital_twin_lifecycle.png", dpi=150, bbox_inches="tight")
plt.close(fig)
for phase, years, focus in phases:
    print(f"{phase:<16} {years:>2} years  focus: {focus}")


Discovery         1 years  focus: resource signal
Appraisal         2 years  focus: resource range
Feasibility       1 years  focus: screen concepts
Concept select    1 years  focus: rank alternatives
FEED              2 years  focus: define project
Detailed design   2 years  focus: freeze scope
Construction      3 years  focus: build and install
Operations       20 years  focus: optimize value


**Discussion (Figure ch11_digital_twin_lifecycle.png).** *Observation.* The lifecycle spends little calendar time before FEED but locks in most value and risk. *Mechanism.* Early phases select reservoir assumptions, concept architecture and host constraints that later engineering can only optimize locally. *Implication.* A digital twin is most valuable before sanction, when alternatives can still be compared cheaply. *Recommendation.* Use lifecycle phase labels in every concept-screen notebook so assumptions are not mixed across decision gates.


## Discipline Integration Map


In [3]:
nodes = {"PVT": (0.12, 0.75), "Reservoir": (0.12, 0.35), "Process": (0.46, 0.75), "Facilities": (0.46, 0.35), "Economics": (0.80, 0.55)}
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis("off")
for name, (x, y) in nodes.items():
    ax.scatter([x], [y], s=2800, c="#d9ecf2", edgecolors="#16425b", linewidths=2)
    ax.text(x, y, name, ha="center", va="center", fontsize=11, fontweight="bold")
arrows = [("PVT", "Process", "fluid props"), ("Reservoir", "Facilities", "rate profile"), ("Process", "Facilities", "duties"), ("Facilities", "Economics", "capex/opex"), ("Economics", "Reservoir", "value feedback")]
for start_name, end_name, label in arrows:
    x1, y1 = nodes[start_name]
    x2, y2 = nodes[end_name]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=2, color="#2f4858"))
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.04, label, ha="center", fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="#cccccc"))
ax.set_title("Digital Field Twin: Data Interfaces Between Disciplines", fontsize=13, fontweight="bold")
fig.savefig(FIGURES_DIR / "ch11_digital_twin_interfaces.png", dpi=150, bbox_inches="tight")
plt.close(fig)
for row in arrows:
    print(f"{row[0]} -> {row[1]}: {row[2]}")


PVT -> Process: fluid props
Reservoir -> Facilities: rate profile
Process -> Facilities: duties
Facilities -> Economics: capex/opex
Economics -> Reservoir: value feedback


**Discussion (Figure ch11_digital_twin_interfaces.png).** *Observation.* The map identifies five domains and five directed data interfaces. *Mechanism.* Each interface passes a small set of engineering variables, such as phase behavior, production rates, duties or cash-flow inputs. *Implication.* Poor interface control causes double counting and inconsistent concept rankings. *Recommendation.* Treat each interface as a contract with units, validity range and owner before running sensitivities.


## Exercises

1. List three variables that move from PVT to process simulation.
2. Explain why concept select should include both reservoir uncertainty and facility capacity.
3. Pick one interface in the map and define an acceptance test for units and ranges.
4. Describe how a higher CO2 fraction propagates through at least three domains.
5. Sketch a minimal results table for a concept-screen digital twin.
